In [ ]:
#install libraries
!pip install gensim
!pip install -q spacy nltk
!python -m spacy download it_core_news_sm

In [ ]:
from google.colab import files #upload module to clean dataset
uploaded = files.upload()



In [ ]:
import clean_emb as em1
import importlib
importlib.reload(em1)

In [ ]:
# create new cleaned column
import pandas as pd

df = pd.read_csv("file.csv")

df["no_stop_words"] = df["comment_message_clean"].apply(em1.clean_text)




This script takes the column containing your cleaned text and breaks every sentence down into a simple list of individual words. Once everything is chopped up into these tokens, it feeds them into the Word2Vec model. By using the skip-gram method, the model looks at each word and learns its context by checking the words immediately surrounding it.

In [ ]:
import re
from collections import Counter
from gensim.models import Word2Vec

column_name = "no_stop_words"
if column_name not in df.columns:
    raise ValueError(f"Column '{column_name}' not found. Available columns: {list(df.columns)}")

# Tokenization
def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zàèéìòùç\s]", " ", text)
    return text.split()

sentences = [tokenize(t) for t in df[column_name].dropna().astype(str)]
tokens = [w for s in sentences for w in s]

counter = Counter(tokens)
print(f"Words in corpus: {len(counter)}")

# Word2Vec training
model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=3,   # ignore very rare words
    sg=1,          # skip-gram
    epochs=30
)
print("Model trained")



This code takes the Word2Vec model trained earlier and looks for the 20 words that are most mathematically similar to your specific target words. The script uses a technique called Principal Component Analysis (PCA) to squash those 100 dimensions down into just 2 dimensions (X and Y). It then creates a scatter plot showing the target word as a red star and its closest neighbors clustered around it.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
import matplotlib.pyplot as plt

def nearest_neighbors_plot(word, k=10):
    word = word.lower().strip()

    # Check if the word is in the model's vocabulary
    if word not in model.wv:
        print(f"Warning: Word '{word}' not found in model vocabulary. Skipping plot.")
        return

    neighbors = model.wv.most_similar(word, topn=k)

    print(f"\nNeighbors of '{word}':")
    for w, score in neighbors:
        print(f"{w}: {score:.4f}")

    # PCA to reduce to 2 dimensions
    neighbor_words = [word] + [w for w, _ in neighbors]
    embeddings = [model.wv[w] for w in neighbor_words]

    # Check to ensure we have enough data for 2 components
    if len(embeddings) < 2:
        print("Not enough neighbors for PCA.")
        return

    # Normalization Step
    # This aligns the vectors on a sphere so PCA respects cosine similarity
    embeddings = normalize(embeddings, norm='l2')

    emb_2d = PCA(n_components=2).fit_transform(embeddings)

    # Plot
    plt.figure(figsize=(12, 10))
    plt.scatter(emb_2d[:, 0], emb_2d[:, 1], s=70, alpha=0.75)

    # Highlight the central word
    plt.scatter(emb_2d[0, 0], emb_2d[0, 1], color="red", s=150, marker="*", label=word)

    # Annotations
    for i, w in enumerate(neighbor_words):
        plt.annotate(
            w,
            xy=(emb_2d[i, 0], emb_2d[i, 1]),       # The point (x, y)
            xytext=(8, 8),                         # Distance from point (x=8, y=8) in points
            textcoords='offset points',            # Interpret xytext as offset in points
            fontsize=10,
            alpha=0.8
        )

    plt.title(f"Target Word: '{word}' and nearest neighbors (Word2Vec)")
    plt.xlabel("PCA 1")
    plt.ylabel("PCA 2")
    plt.legend()

    plt.xlim(-0.5, 0.5)
    plt.ylim(-0.5, 0.5)
    plt.tight_layout()

    # Save to PDF
    filename = f"pca_{word}_1.pdf"
    plt.savefig(filename, format='pdf', bbox_inches='tight')
    print(f"Plot saved successfully as: {filename}")

    # Show the plot
    plt.show()

    # Close the figure to free memory
    plt.close()

# Running the function for the requested words
nearest_neighbors_plot("shoah", k=20)
nearest_neighbors_plot("gaza", k=20)
nearest_neighbors_plot("israele", k=20)
nearest_neighbors_plot("palestinese", k=20)
nearest_neighbors_plot("ebreo", k=20)
nearest_neighbors_plot("ebrei", k=20)
nearest_neighbors_plot("palestinesi", k=20)